# Chapter 18. Efficient Fine-Tuning of LLMs with LoRA and QLoRA

In [1]:
!pip install -q -U bitsandbytes peft trl vllm openai
!pip uninstall -q -y torchao

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompati

## 18.2 Learning Objectives

By the end of this chapter, you should be able to:

- configure four-bit NF4 quantization;
- configure LoRA using PEFT;
- prepare conversational prompt-completion training data;
- inspect the sequence-length distribution before choosing a training length;
- configure and run supervised fine-tuning using `SFTTrainer`.

## 18.3 Prompting Before Fine-Tuning

Suppose we need an LLM to perform a specialized task. One possible workflow is to collect the data and immediately use it to fine-tune a model. But that approach skips a crucial question: what if the instruction model can already handle the task well? For example, an instruction model may already understand requests such as "*return the result as JSON*" or "*Classify this request using one of the following categories*". Before training anything, we should establish a reasonable prompt and measure the resulting behavior.

### 18.3.1 Why a Baseline Matters

Suppose prompting achieves 92% on the metric that matters for your application, while fine-tuning achieves 93%. Technically, fine-tuning improved the model, but a one-point gain may not justify:
- maintaining training data;
- running a training pipeline;
- versioning adapters;
- evaluating new adapter versions;
- deploying another model artifact.

Now imagine a situation where prompting only delivers 62% while fine-tuning reaches 91%. Now it makes sense to deploy a fine-tuned model. But, without a baseline, we cannot make that distinction.

### 18.3.2 Keep the Experiment Controlled

Ideally, before and after fine-tuning we keep constant:

```
same instruction checkpoint
same tokenizer
same chat template
same held-out examples
same decoding parameters
same evaluation procedure
```

The major difference is just the model itself: original vs fine-tuned.

### 18.3.3 Task-Specific Fine-Tuning

Typically, our starting checkpoint has already been instruction-tuned. Fine-tuning becomes interesting when we want a behavior to become more:
- reliable;
- consistent;
- specialized;
- compact;
- difficult to obtain through prompting alone.

## 18.4 Why Full Fine-Tuning Is Expensive

Our model contains roughly 360 million parameters. Let's first consider only parameter storage. If we're using FP32, the model requires approximately 1.44 Gb. If we load it using FP16 or BF16 instead, it will be around 720 Mb. But the fine-tuning process requires much more than model weights. We may also need memory for:
- gradients
- optimizer state
- activations
- temporary tensors

If every parameter is trainable, every parameter may need a gradient. Adam-like optimizers maintain two additional values associated with each trainable parameter. So, for every parameter, we already have three other values associated with it. Therefore, we're already at 5.76 Gb for FP32 or 2.88 Gb for either FP16 or BF16. So training memory can be several times greater than inference memory.

Backpropagation also needs intermediate results from the forward pass. Activation memory grows with factors such as batch size, sequence length, the dimensionality of the embeddings, and the number of layers. For multi-billion-parameter models, full fine-tuning quickly becomes impractical on consumer hardware. So, we'll attack the problem in two ways:
- quantization will reduce the weight's memory footprint
- low-rank adapters (LoRA) will greatly reduce the number of trainable parameters

## 18.5 Quantization

Quantization represents model values using fewer bits. Some common representations are:

| Representation | Bits | Bytes |
| -------------- | ---- | ----- |
| FP32           | 32   | 4     |
| FP16/BF16      | 16   | 2     |
| INT8           | 8    | 1     |
| 4-bit          | 4    | 0.5   |

Moving from 16-bit storage to four-bit storage can therefore reduce the raw storage requirement substantially. There is no free lunch, though, as quantization is an approximation. A four-bit representation has only 16 possible bit patterns, so we can't independently represent every possible floating-point value. The model trades numerical precision for reduced memory and memory bandwidth.

Moreover, there is a crucial distinction between storage and compute precision. Weights may be stored using four bits but temporarily interpreted at FP16 or BF16 precision during computation, and that's exactly what we'll do.

## 18.6 Setting Up the Model

In [2]:
import math
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime for this chapter.")

print(torch.cuda.get_device_name(0))

Tesla T4


In [3]:
checkpoint = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.unk_token

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

### 18.6.1 Configuring Four-Bit Quantization

In [4]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
compute_dtype

torch.float16

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    quantization_config=bnb_config,
    dtype=compute_dtype,
    device_map={"": 0}
)
model.eval()

model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 960, padding_idx=2)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=960, out_features=960, bias=False)
          (k_proj): Linear4bit(in_features=960, out_features=320, bias=False)
          (v_proj): Linear4bit(in_features=960, out_features=320, bias=False)
          (o_proj): Linear4bit(in_features=960, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear4bit(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear4bit(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

In [7]:
model.get_memory_footprint() / 1024**2

240.1192626953125

## 18.7 Loading the Training Dataset

In [8]:
from datasets import load_dataset

dataset = load_dataset("trl-lib/Capybara")
dataset["train"][0]

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 37.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  482kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15806 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

{'source': 'GPT4LLM',
 'messages': [{'content': 'Recommend a movie to watch.\n', 'role': 'user'},
  {'content': 'I would recommend the movie, "The Shawshank Redemption" which is a classic drama film starring Tim Robbins and Morgan Freeman. This film tells a powerful story about hope and resilience, as it follows the story of a young man who is wrongfully convicted of murder and sent to prison. Amidst the harsh realities of prison life, the protagonist forms a bond with a fellow inmate, and together they navigate the challenges of incarceration, while holding on to the hope of eventual freedom. This timeless movie is a must-watch for its moving performances, uplifting message, and unforgettable storytelling.',
   'role': 'assistant'},
  {'content': 'Describe the character development of Tim Robbins\' character in "The Shawshank Redemption".',
   'role': 'user'},
  {'content': 'In "The Shawshank Redemption", Tim Robbins plays the character of Andy Dufresne, a banker who is wrongfully con

### 18.7.1 Selecting Single-Turn Examples

In [9]:
def is_single_turn(example):
    messages = example["messages"]
    return (len(messages) == 2 and messages[0]["role"] == "user" and messages[1]["role"] == "assistant")

single_train = dataset["train"].filter(is_single_turn)
single_test = dataset["test"].filter(is_single_turn)

Filter:   0%|          | 0/15806 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

### 18.7.2 Building Prompt-Completion Examples

In [10]:
def to_prompt_completion(example):
    return {"prompt": [example["messages"][0]], "completion": [example["messages"][1]]}

train_data = single_train.map(to_prompt_completion, remove_columns=single_train.column_names)
test_data = single_test.map(to_prompt_completion, remove_columns=single_test.column_names)

example = train_data[0]

Map:   0%|          | 0/5258 [00:00<?, ? examples/s]

Map:   0%|          | 0/67 [00:00<?, ? examples/s]

In [11]:
conversation = example["prompt"] + example["completion"]
example, conversation

({'prompt': [{'content': 'Determine the result obtained by evaluating 5338245-50629795848152. Numbers and symbols only, please.',
    'role': 'user'}],
  'completion': [{'content': '5338245 - 50629795848152 = -50629790509907',
    'role': 'assistant'}]},
 [{'content': 'Determine the result obtained by evaluating 5338245-50629795848152. Numbers and symbols only, please.',
   'role': 'user'},
  {'content': '5338245 - 50629795848152 = -50629790509907',
   'role': 'assistant'}])

In [12]:
print(tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False))

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Determine the result obtained by evaluating 5338245-50629795848152. Numbers and symbols only, please.<|im_end|>
<|im_start|>assistant
5338245 - 50629795848152 = -50629790509907<|im_end|>



## 18.8 Inspecting Sequence Lengths

In [13]:
def formatted_length(example):
    messages = example["prompt"] + example["completion"]

    return len(tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)['input_ids'])

train_lengths = [formatted_length(example) for example in train_data]
test_lengths = [formatted_length(example) for example in test_data]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (20166 > 8192). Running this sequence through the model will result in indexing errors


In [17]:
train_stats = pd.Series(train_lengths, name='Train').describe(percentiles=[.50, .90, .95, .99])
test_stats = pd.Series(test_lengths, name='Test').describe(percentiles=[.50, .90, .95, .99])
pd.DataFrame([train_stats, test_stats])

,count,mean,std,min,50%,90%,95%,99%,max
Train,5258.0,373.620198,473.725843,39.0,310.0,689.3,822.15,1424.72,20166.0
Test,67.0,400.895522,408.104418,60.0,340.0,614.6,694.50,2343.60,2667.0


### 18.8.1 Removing Long Sequences

In [18]:
max_length = 1024
train_data = train_data.filter(lambda example: formatted_length(example) <= max_length)
test_data = test_data.filter(lambda example: formatted_length(example) <= max_length)
len(train_data), len(test_data)

Filter:   0%|          | 0/5258 [00:00<?, ? examples/s]

Filter:   0%|          | 0/67 [00:00<?, ? examples/s]

(5142, 65)

### 18.9.2 Subsetting the Dataset

In [19]:
n_train = min(1000, len(train_data))
n_test = min(100, len(test_data))
train_data = train_data.shuffle(seed=42).select(range(n_train))
test_data = test_data.shuffle(seed=42).select(range(n_test))

## 18.9 Establishing the Prompting Baseline

In [20]:
def generate_response(model, prompt_messages, max_new_tokens=128):
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    prompt_length = inputs["input_ids"].shape[-1]

    model.eval()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    response_ids = outputs[0, ]

    return tokenizer.decode(response_ids, skip_special_tokens=True).strip()

### 18.9.1 Fixed Demonstration Set

In [21]:
demo_data = test_data.select(range(min(8, len(test_data))))

baseline_responses = []

for example in demo_data:
    response = generate_response(model, example["prompt"])
    baseline_responses.append(response)
    print("PROMPT")
    print(example["prompt"][0]["content"])
    print("\nREFERENCE")
    print(example["completion"][0]["content"])
    print("\nPROMPTING BASELINE")
    print(response)
    print("\n" + "=" * 70)

PROMPT
Below is an instruction that describes a task. Write a response that appropriately completes the request.
Use the information in the following paragraphs to answer the question at the end. Explain the answer with reference to these paragraphs. You should prefer to use these paragraphs to answer the question but can use additional outside information only if required to fully answer the question. If you don't know, say that you do not know.

  The mediaTitle is a string field and accepts values within ^[^\n]+ - (s[0-9]+e([0-9]+|null)|movie|short-form|sports|promotional|supplement|not provided) - [a-fA-F0-9]{8}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{12}$. The title of the media being played. internalTitle for movies and series collectionKey for collections.

  Can you provide an example of a valid string for mediaTitle that passes it's regex validation?

REFERENCE
Here's an example of a valid string for mediaTitle that passes its regex validation:

ExampleMovie -

## 18.10 Measuring Baseline Completion Loss

In [22]:
def encode_prompt_completion(example):
    prompt_ids = tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=True,
        add_generation_prompt=True,
    )['input_ids']

    full_ids = tokenizer.apply_chat_template(
        (example["prompt"] + example["completion"]),
        tokenize=True,
        add_generation_prompt=False,
    )['input_ids']

    if (full_ids[:len(prompt_ids)] != prompt_ids):
        raise ValueError("The prompt template does not match the beginning of the full conversation.")

    labels = ([-100] * len(prompt_ids) + full_ids[len(prompt_ids):])

    return (full_ids, labels)

### 18.10.1 Evaluation Helper

In [23]:
def completion_loss(model, dataset, batch_size=8,):
    total_loss = 0.0
    total_tokens = 0

    model.eval()

    for start in range(0, len(dataset), batch_size):
        batch = dataset[start:start + batch_size]
        encoded = [encode_prompt_completion({"prompt": batch["prompt"][i], "completion": batch["completion"][i]})
            for i in range(len(batch["prompt"]))]

        max_len = max(len(input_ids) for input_ids, _ in encoded)

        input_rows = []
        attention_rows = []
        label_rows = []

        for input_ids, labels in encoded:
            padding = max_len - len(input_ids)
            input_rows.append(input_ids + [tokenizer.pad_token_id] * padding)
            attention_rows.append([1] * len(input_ids) + [0] * padding)
            label_rows.append(labels + [-100] * padding)

        input_ids = torch.tensor(input_rows, device=model.device)
        attention_mask = torch.tensor(attention_rows, device=model.device)
        labels = torch.tensor(label_rows, device=model.device)
        n_tokens = (labels != -100).sum().item()

        with torch.inference_mode():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        total_loss += (outputs.loss.item() * n_tokens)
        total_tokens += n_tokens

    return (total_loss / total_tokens)

In [24]:
baseline_loss = completion_loss(model, test_data)
baseline_perplexity = math.exp(baseline_loss)
baseline_metrics = {"completion_loss": baseline_loss, "perplexity": baseline_perplexity}
baseline_metrics

{'completion_loss': 1.241572351660698, 'perplexity': 3.461051178211311}

## 18.11 Parameter-Efficient Fine-Tuning (PEFT)

Parameter-Efficient Fine-Tuning, or **PEFT**, keeps most pretrained parameters frozen and trains a relatively small number of additional parameters.

Full fine-tuning changes the original model parameters themselves. PEFT takes a different approach: it keeps the expensive pretrained model largely fixed and learns a much smaller task-specific update. This reduces the number of trainable parameters, optimizer states, and gradients, while preserving the knowledge already encoded in the base model.

There are several PEFT techniques. In this chapter we'll focus on LoRA because it is widely used for adapting Transformer models and combines naturally with four-bit quantization.

### 18.11.1 Low-Rank Adaptation (LoRA)

Suppose a pretrained linear layer has a weight matrix $W$. Full fine-tuning would learn an update $\Delta W$ with the same dimensions as $W$. LoRA instead approximates that update as the product of two much smaller trainable matrices:

$$
\Delta W = BA
$$

If $W$ maps $d_{in}$ inputs to $d_{out}$ outputs, we can choose $A$ with shape $r \times d_{in}$ and $B$ with shape $d_{out} \times r$, where $r$ is much smaller than either original dimension. The effective layer becomes:

$$
y = Wx + sBAx
$$

The original $W$ stays frozen; only $A$ and $B$ are trained. This is where most of LoRA's parameter savings come from.

### 18.11.2 Rank

The rank $r$ controls the size of the low-rank update. A larger rank gives the adapter more capacity because $A$ and $B$ can represent a richer set of changes, but it also increases the number of trainable parameters and memory usage. Very small ranks are cheap but may be too restrictive for some tasks.

There is no universally optimal rank. Values such as 8, 16, 32, or 64 are common starting points. We'll use a rank of 16, which is modest for this model and sufficient for illustrating the method.

### 18.11.3 Alpha

`lora_alpha` controls the scale of the adapter update. PEFT applies a scaling factor related to $\alpha / r$ to the LoRA contribution, so alpha determines how strongly the learned low-rank update influences the frozen base layer.

We'll use `lora_alpha=32` with `r=16`, giving the adapter update a scaling factor of two under the standard LoRA formulation. Rank and alpha should be considered together rather than interpreted as completely independent parameters.

### 18.11.4 Dropout

LoRA can also apply dropout to the input of the adapter branch during training. This does not drop weights from the frozen base model; it regularizes the learned LoRA update. A small value such as 0.05 can help reduce overfitting, especially when adapting a model with a relatively small dataset. Dropout is disabled automatically during evaluation and inference.

## 18.12 Preparing the Quantized Model for Training

In [25]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

### 18.12.1 Configuring LoRA

In [26]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    exclude_modules=["lm_head"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

In [27]:
# for Colab only (without BF16) - turns parameters into FP32
if compute_dtype == torch.float16:
  for param in model.parameters():
      if param.requires_grad:
          param.data = param.data.float()

In [28]:
model.print_trainable_parameters()

trainable params: 8,683,520 || all params: 370,504,640 || trainable%: 2.3437


## 18.13 Configuring Supervised Fine-Tuning

With the model prepared, the next step is to configure supervised fine-tuning. `SFTTrainer` builds on the Hugging Face Trainer infrastructure but understands common language-model fine-tuning formats, including conversational prompt-completion datasets such as ours.
The configuration below is intentionally conservative for a free Colab GPU: a small per-device batch, gradient accumulation, gradient checkpointing, one epoch, and a maximum length of 1,024 tokens.

### 18.13.1 Completion-Only Loss

Because our dataset explicitly separates `prompt` and `completion`, we can train only on the completion tokens. With `completion_only_loss=True`, prompt tokens provide context to the model but do not contribute to the supervised loss. This matches the objective we used for the baseline: given the prompt, make the desired assistant response more likely.

### 18.13.2 Gradient Accumulation

A physical batch size of two is small, but increasing it directly would consume more GPU memory. Gradient accumulation lets us approximate a larger batch by running several forward/backward passes before applying an optimizer update. With `per_device_train_batch_size=2` and `gradient_accumulation_steps=4`, each optimizer step incorporates gradients from an effective batch of eight examples on a single GPU.

### 18.13.3 Gradient Checkpointing

Gradient checkpointing addresses activation memory rather than parameter memory. Normally, backpropagation stores many intermediate activations from the forward pass. With checkpointing, some of those activations are discarded and recomputed during the backward pass. This reduces memory usage at the cost of additional computation, which is often a worthwhile trade on a memory-constrained GPU.

In [34]:
from trl import SFTConfig, SFTTrainer

major, minor = torch.cuda.get_device_capability()

training_args = SFTConfig(
    output_dir="smollm2-360m-capybara-qlora",
    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,
    gradient_checkpointing=True,

    learning_rate=2e-4,
    warmup_steps=3,

    max_length=max_length,

    completion_only_loss=True,

    eval_strategy="epoch",
    save_strategy="no",

    logging_steps=10,

    fp16=(compute_dtype == torch.float16) if major >= 8 else False,
    bf16=(compute_dtype == torch.bfloat16),

    report_to="none",
    seed=42,
)

## 18.14 Building the Trainer

In [35]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    processing_class=tokenizer,
)

In [36]:
print("Training examples:", len(train_data))
print("Evaluation examples:", len(test_data))

Training examples: 1000
Evaluation examples: 65


### 18.14.1 Training

In [37]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.282384,1.098481,1.091375,0.712682,342592.000000


In [38]:
train_result.metrics

{'train_runtime': 457.5738,
 'train_samples_per_second': 2.185,
 'train_steps_per_second': 0.273,
 'total_flos': 918164380880640.0,
 'train_loss': 1.187613136291504,
 'epoch': 1.0}

In [39]:
trainer_metrics = trainer.evaluate()
trainer_metrics

Training Loss,Validation Loss,Epoch,Entropy,Mean Token Accuracy,Num Tokens
1.282384,1.098481,1,1.091375,0.712682,342592.000000


{'eval_loss': 1.0984805822372437,
 'eval_entropy': 1.091375391591679,
 'eval_mean_token_accuracy': 0.7126820195804943,
 'eval_num_tokens': 342592.0}

## 18.15 Evaluating the Fine-Tuned Model

In [40]:
model.config.use_cache = True
model.eval()

fine_tuned_loss = completion_loss(model, test_data)
fine_tuned_perplexity = math.exp(fine_tuned_loss)

loss_comparison = pd.DataFrame({
    "Prompted Instruct": {"completion_loss": baseline_loss, "perplexity": baseline_perplexity},
    "QLoRA": {"completion_loss": fine_tuned_loss, "perplexity": fine_tuned_perplexity},
})
loss_comparison

,Prompted Instruct,QLoRA
completion_loss,1.241572,1.134013
perplexity,3.461051,3.108103


In [41]:
loss_reduction = baseline_loss - fine_tuned_loss
relative_loss_reduction = loss_reduction / baseline_loss

print("Absolute loss reduction:", loss_reduction)
print("Relative loss reduction:", f"{relative_loss_reduction:.2%}")

Absolute loss reduction: 0.10755967161458013
Relative loss reduction: 8.66%


### 18.15.1 Comparing the Generated Responses

In [42]:
fine_tuned_responses = []

for example in demo_data:
    response = generate_response(model, example["prompt"])
    fine_tuned_responses.append(response)

response_comparison = pd.DataFrame({
    "Prompt": [example["prompt"][0]["content"] for example in demo_data],
    "Reference": [example["completion"][0]["content"] for example in demo_data],
    "Prompted Instruct": baseline_responses,
    "QLoRA": fine_tuned_responses})

for row in response_comparison.itertuples():
    print("PROMPT")
    print(row.Prompt)
    print("\nREFERENCE")
    print(row.Reference)
    print("\nPROMPTED INSTRUCT")
    print(row[response_comparison.columns.get_loc("Prompted Instruct")])
    print("\nQLORA")
    print(row.QLoRA)
    print("\n" + "=" * 70)

PROMPT
Below is an instruction that describes a task. Write a response that appropriately completes the request.
Use the information in the following paragraphs to answer the question at the end. Explain the answer with reference to these paragraphs. You should prefer to use these paragraphs to answer the question but can use additional outside information only if required to fully answer the question. If you don't know, say that you do not know.

  The mediaTitle is a string field and accepts values within ^[^\n]+ - (s[0-9]+e([0-9]+|null)|movie|short-form|sports|promotional|supplement|not provided) - [a-fA-F0-9]{8}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{12}$. The title of the media being played. internalTitle for movies and series collectionKey for collections.

  Can you provide an example of a valid string for mediaTitle that passes it's regex validation?

REFERENCE
Here's an example of a valid string for mediaTitle that passes its regex validation:

ExampleMovie -

## 18.16 Fine-Tuning May Not Always Win

There are several possible outcomes. There could be a large improvement, making fine-tuning easy to justify; or a small improvement, Whether the adapter is worth maintaining becomes a product or engineering decision; or perhaps no improvement or even degradation of the performance.

Possible explanations for the lack of improvement include:
- the instruction model already solves the task well;
- the dataset is too small;
- the dataset doesn't match the evaluation task;
- the learning rate is poor;
- the adapter rank is inadequate;
- the prompt already communicates the behavior effectively.

In those case where fine-tuning make behavior worse, possible reasons include:
- overfitting;
- noisy examples;
- excessive learning rate;
- too many epochs;
- a mismatch between training and evaluation data.

This is why baseline evaluation isn't optional.

## 18.17 Saving the LoRA Adapter

In [43]:
import os

adapter_dir = "smollm2-360m-capybara-qlora-adapter"

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

for filename in sorted(os.listdir(adapter_dir)):
    path = os.path.join(adapter_dir, filename)

    if os.path.isfile(path):
        print(filename, f"{os.path.getsize(path) / 1024**2:.2f} MB")

README.md 0.00 MB
adapter_config.json 0.00 MB
adapter_model.safetensors 16.62 MB
chat_template.jinja 0.00 MB
tokenizer.json 3.36 MB
tokenizer_config.json 0.00 MB


## 18.18 Reloading the Adapter

In [44]:
import gc

del trainer
del model
gc.collect()
torch.cuda.empty_cache()

instruction_model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    quantization_config=bnb_config,
    dtype=compute_dtype,
    device_map={"": 0}
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [45]:
from peft import PeftModel

reloaded_model = PeftModel.from_pretrained(instruction_model, adapter_dir)
reloaded_model.config.use_cache = True
reloaded_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 960, padding_idx=2)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=960, out_features=960, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=960, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=960, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_p

In [46]:
reloaded_loss = completion_loss(reloaded_model, test_data)
print("Fine-tuned loss:", fine_tuned_loss)
print("Reloaded loss:", reloaded_loss)
print("Difference:", abs(fine_tuned_loss - reloaded_loss))

Fine-tuned loss: 1.1340126800461179
Reloaded loss: 1.1339950336506412
Difference: 1.7646395476633714e-05


In [47]:
reloaded_responses = []

for example in demo_data:
    response = generate_response(reloaded_model, example["prompt"])
    reloaded_responses.append(response)

reload_match_rate = sum(before == after for before, after in zip(fine_tuned_responses, reloaded_responses)) / len(fine_tuned_responses)
reload_match_rate

1.0

## 18.19 Managing Adapters

During inference, LoRA normally keeps both the base model and the adapter as separate components. This has several advantages:
- small adapter files;
- one base model can support several tasks;
- adapters can be replaced independently;
- the original model remains unchanged.

But it's also possible to merge a LoRA update into a compatible higher-precision model using PEFT's `merge_and_uload()` method. For a quantized model, however, merging directly into the four-bit representation isn't always the appropriate deployment path.

A common approach is to reload the base model at FP16/BF16, load the adapter, and then merge it and save the merged model. Whether to merge depends on deployment requirements. For this course, we'll keep the adapter separate.

### 18.19.1 One Model, Multiple Adapters

Suppose the same instruction model must support different tasks, such as customer-support behavior and SQL generation. Full fine-tuning could require two complete model copies. With LoRA, we can instead have multiple adapters on top of the same base model. This way, the expensive underlying weights remain shared. This is one reason PEFT can be attractive even when full fine-tuning would technically fit in memory.

## 18.20 Common Fine-Tuning Failure Modes

Here is a non-exhaustive list of common failure modes:
- weak prompting baseline: If the baseline prompt is deliberately poor, fine-tuning will appear artificially impressive. Use a reasonable prompt first.
- training/evaluation mismatch: If training examples represent one behavior and evaluation measures another, fine-tuning results can be misleading.
- wrong chat template: The conversation format used during training should match what the model sees during inference.
- poor training data: Fine-tuning reinforces patterns in the examples. Bad responses produce bad supervision.
- excessive learning rate:  A very high learning rate can destabilize or over-specialize the adapter.
- too many epochs: Repeatedly training on a small dataset can produce memorization rather than generalization.
- evaluating the training loss only: The training loss tells us how well the model fit its training examples, but it does not tell us whether the model improved on new examples. That's why we reserved a held-out evaluation set.

## 18.21 Prompting, Retrieval, or Fine-Tuning?

Part IV has given us three different tools, prompting, retrieval, and fine-tuning. Use prompting when the instruction model can already perform the desired behavior reliably after receiving clear instructions. It has clear advantages: no training is needed, it is suitable for fast experimentation, and behavior can be changed immediately. Use retrieval when the missing ingredient is information external to the model. It is especially appropriate when information changes, needs traceability, must remain externally manageable, or is too large to encode conveniently into model parameters.

Use fine-tuning when the model's behavior itself should change. Examples include:
- specialized output schemas;
- consistent formatting;
- domain-specific transformations;
- task-specific response patterns;
- behavior that prompting produces inconsistently.

## 18.22 Serving Fine-Tuned Models with vLLM

So far, we've used Transformers, PEFT, and TRL packages for both training and inference. That's convenient during development, but once a model becomes part of an application, inference has different requirements. Instead of processing one notebook request at a time, a serving system may need to:
- accept concurrent requests
- batch them efficiently
- manage the KV cache
- stream generated tokens
- share one base model across adapters
- expose an HTTP API

This is where **vLLM** becomes useful. vLLM is an inference and serving engine designed specifically for language models. It includes techniques such as **PagedAttention**, continuous batching, prefix caching, optimized attention/GEMM kernels, and high-throughput request scheduling. It also provides an OpenAI-compatible HTTP server.

### 18.22.1 What Happens to Our QLoRA Model?

During training, we used a 4-bit instruction model with an attached trainable LoRA adapter. The four-bit representation helped us fit training into limited GPU memory. The artifact we saved was the **LoRA adapter** only, as it is separate from the quantized base-model representation.

For deployment, we can use whatever inference configuration makes sense for the deployment hardware. We could serve the original model at FP16/BF16 with the adapter attached, or choose a supported quantized inference format independently. vLLM supports several quantization approaches, including BitsAndBytes, GPTQ, AWQ, and other formats, although hardware support varies among them.

For our small 360M model, there's little reason to complicate the first serving example with additional quantization, though.

### 18.22.2 Offline Inference with a LoRA Adapter

In [52]:
import os

# vLLM is being used inside a notebook after CUDA has already been initialized.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
# Release the Transformers/PEFT models before vLLM allocates its own GPU memory.
del reloaded_model
del instruction_model
gc.collect()
torch.cuda.empty_cache()

In [53]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(model=checkpoint, enable_lora=True, max_lora_rank=16, dtype=compute_dtype)

(MainProcess pid=1557) ERROR 11-11 19:58:18 [fa_utils.py:212] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


[rank0]:W1111 19:59:23.474000 1557 torch/_inductor/utils.py:1806] Not enough SMs to use max_autotune_gemm mode
Capturing CUDA graphs (FULL): 100%|██████████| 70/70 [00:08<00:00,  8.38it/s]


In [54]:
sampling_params = SamplingParams(temperature=0.0, max_tokens=128)

In [55]:
messages = [
    {
        "role": "user",
        "content": "Explain gradient clipping in one short paragraph.",
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [57]:
outputs = llm.generate(
    [prompt],
    sampling_params,
    lora_request=LoRARequest("capybara", 1, adapter_dir),
)
print("\n", outputs[0].outputs[0].text)

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.05s/it, est. speed input: 12.52 toks/s, output: 35.91 toks/s]


 Gradient clipping is a technique used in deep learning to prevent the gradient of a model's output from becoming too large or too small, which can cause the model to converge to a suboptimal solution. It works by clipping the gradient of the model's output to a certain range, usually between -1 and 1, to prevent the model from becoming unstable or unstable. This helps to prevent the model from "over-optimizing" or "under-optimizing" the model's weights, and can improve the model's stability and convergence.


### 18.22.3 Serving the Model Through an API

For an application, we usually don't want Python code to instantiate the model directly. Instead, we can start a model server.

```bash
vllm serve HuggingFaceTB/SmolLM2-360M-Instruct \
    --enable-lora \
    --max-lora-rank 16 \
    --lora-modules capybara=./smollm2-360m-capybara-qlora-adapter
```

In [1]:
import os
import subprocess
import time
import requests

adapter_path = os.path.abspath("smollm2-360m-capybara-qlora-adapter")
assert os.path.isdir(adapter_path), adapter_path

log_path = "/tmp/vllm_server.log"
log_file = open(log_path, "w")

server = subprocess.Popen(
    [
        "vllm", "serve",
        "HuggingFaceTB/SmolLM2-360M-Instruct",
        "--dtype", "float16",
        "--enforce-eager",
        "--max-model-len", "1024",
        "--gpu-memory-utilization", "0.4",
        "--enable-lora",
        "--max-lora-rank", "16",
        "--lora-modules", f"capybara={adapter_path}",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

In [2]:
import time
import requests

for _ in range(120):
    if server.poll() is not None:
        print(f"vLLM exited with code {server.returncode}")
        break

    try:
        response = requests.get("http://localhost:8000/health", timeout=1)

        if response.ok:
            print("vLLM server is ready.")
            break
    except requests.RequestException:
        pass

    time.sleep(2)
else:
    print("Server did not become ready.")

if server.poll() is not None:
    log_file.flush()

    with open(log_path) as f:
        lines = f.readlines()

    print("".join(lines[-100:]))

vLLM server is ready.


In [3]:
!curl http://localhost:8000/v1/models

{"object":"list","data":[{"id":"HuggingFaceTB/SmolLM2-360M-Instruct","object":"model","created":1794428897,"owned_by":"vllm","root":"HuggingFaceTB/SmolLM2-360M-Instruct","parent":null,"max_model_len":1024,"permission":[{"id":"modelperm-9cfff58337980cd9","object":"model_permission","created":1794428897,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]},{"id":"capybara","object":"model","created":1794428897,"owned_by":"vllm","root":"/content/smollm2-360m-capybara-qlora-adapter","parent":"HuggingFaceTB/SmolLM2-360M-Instruct","max_model_len":null,"permission":[{"id":"modelperm-b89d8fc3f0e0a952","object":"model_permission","created":1794428897,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

The server exposes an HTTP API. vLLM supports an OpenAI-compatible interface, including:

```
/v1/completions
/v1/chat/completions
/v1/responses
```

The adapter also appears as a model that clients can select. For example, the following query will list both the underlying model and configured LoRA adapters.

```
curl http://localhost:8000/v1/models
```

### 18.22.4 Using the OpenAI Python Client

In [4]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
response = client.chat.completions.create(
        model="capybara",
        messages=[
            {"role": "user",
             "content": "Explain gradient clipping in one short paragraph."}
        ],
        temperature=0,
        max_tokens=128,
)
print("\n", response.choices[0].message.content)


 Gradient clipping is a technique used in deep learning to prevent the gradient of a model's output from becoming too large or too small, which can cause the model to converge to a suboptimal solution. It works by clipping the gradient of the model's output to a certain range, usually between -1 and 1, to prevent the model from becoming unstable or unstable. This helps to prevent the model from "over-optimizing" or "under-optimizing" the model's weights, and can improve the model's stability and convergence.


### 18.22.5 Why vLLM Instead of `model.generate()`?

For a single notebook request, `model.generate()` is perfectly reasonable. The difference becomes important when many users are generating concurrently. While a simple application might process requests sequentially, that leaves opportunities for GPU capacity to go unused. vLLM can schedule requests dynamically using **continuous batching**. As requests arrive and finish, the active batch can change.

vLLM also uses **PagedAttention** to manage KV-cache memory efficiently instead of requiring every sequence's cache to occupy one large contiguous allocation. Its current engine also includes prefix caching, chunked prefill, optimized kernels, and other inference-oriented optimizations.

## 18.23 Beyond Supervised Fine-Tuning

Supervised fine-tuning isn't the only form of post-training. Modern LLM development may also include:
- preference optimization;
- reward modeling;
- reinforcement-learning approaches.

Those methods address questions beyond finding the response that follows a given prompt. For example, preference methods can learn from comparisons between multiple possible responses. Although, those techniques are outside the scope of this course, the foundations you've learned here remain relevant: instruction data, chat templates, causal language modeling, quantization, PEFT, adapters, etc.